# GridSense-AZ — GWNet training on Colab GPU

**Target:** beat the local CPU `gwnet_v0` (5 epochs, batch=64, lr=2e-3) with a longer GPU run.

**Payload:** upload `gridsense_payload.tar.gz` (5 MB, staged at `/tmp/gridsense_payload.tar.gz` on the Jarvis machine) via Colab's left-panel file upload, OR drop it into Drive and `cp` it over.

**Runtime:** T4 or L4 GPU. A100 optional.

**Entry:** `scripts/train.py --start 2022-06-01 --end 2023-10-01 --epochs 30 --batch-size 128 --lr 1e-3 --device cuda`.

Outputs (`gwnet_v0.pt`, `metrics.json`, `history.json`) are zipped to `/content/gwnet_v1.zip` at the end for easy download.

## 1. Runtime check

In [ ]:
!nvidia-smi

## 2. Download payload

Pulls the 5 MB `gridsense_payload.tar.gz` directly from the github repo — no manual upload needed.


In [ ]:
# Auto-download payload from the GridSense-AZ github repo.
!wget -q -O /content/gridsense_payload.tar.gz https://raw.githubusercontent.com/dc-ai-labs/gridsense-az/main/notebooks/gridsense_payload.tar.gz
!ls -la /content/gridsense_payload.tar.gz && echo '--- sha256 ---' && sha256sum /content/gridsense_payload.tar.gz


In [ ]:
!rm -rf /content/gridsense-az && mkdir -p /content/gridsense-az
!tar -xzf /content/gridsense_payload.tar.gz -C /content/gridsense-az
!ls /content/gridsense-az
!ls /content/gridsense-az/data/raw/noaa | head
!ls /content/gridsense-az/data/raw/eia930
!ls /content/gridsense-az/data/ieee123 | head

## 3. Install dependencies

Colab already ships with torch, numpy, pandas, pyarrow. We only pin the missing pieces.

In [ ]:
!pip install -q torch-geometric networkx opendssdirect.py

In [ ]:
# sanity
import torch, torch_geometric, pandas, numpy, pyarrow, networkx, opendssdirect
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('pyg', torch_geometric.__version__)
print('pandas', pandas.__version__)
print('networkx', networkx.__version__)

## 4. Smoke test (1 epoch, tiny window)

Per ML-Trainer protocol rule #6: no full run without a dry run first.

In [ ]:
%cd /content/gridsense-az
!python scripts/train.py \
  --start 2023-07-01 --end 2023-07-15 \
  --epochs 1 --batch-size 64 --lr 1e-3 \
  --device cuda --out-dir /tmp/smoke 2>&1 | tail -40

## 5. Full run

If smoke test prints a final `test MAE` row without a traceback, launch the full job.

In [ ]:
%cd /content/gridsense-az
!mkdir -p data/models
!python scripts/train.py \
  --start 2022-06-01 --end 2023-10-01 \
  --epochs 200 --batch-size 128 --lr 2e-3 \
  --scheduler cosine --warmup-epochs 10 \
  --device cuda --out-dir data/models 2>&1 | tee /content/train.log

## 6. Package outputs for download

In [ ]:
!cd /content/gridsense-az/data/models && ls -lh && cat metrics.json
!cd /content/gridsense-az && zip -j /content/gwnet_v1.zip data/models/gwnet_v0.pt data/models/metrics.json data/models/history.json /content/train.log
!ls -lh /content/gwnet_v1.zip

In [ ]:
from google.colab import files
files.download('/content/gwnet_v1.zip')